In [ ]:
#Summary: This script is designed to forced align audio files containing multilingual speech (read passages).
#Method: CTC forced alignment using torchaudio

#Set-up
import torch
import torchaudio
import IPython
import matplotlib.pyplot as plt
import torchaudio.functional as F
from typing import List
import re
from pathlib import Path
import os
import csv
import numpy as np
import unicodedata

import ipdb #debug functionality

print("Torchaudio version:", torchaudio.__version__)
print("Available backends:", torchaudio.list_audio_backends())

In [ ]:
#Check computing architecture -- CUDA (Compute Unified Device Architecutre allows use of GPUs 
#for general purpose processing; only supported for NVIDIA chips) or CPU -- and set device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
#Instantiate the packages necessary to process the waveforms: a pre-trained acoustic model, 
#a tokenizer that uses the same set of tokens as the model, and an aligner.  Installs draw on MMS_FA
#which is a pre-trained Wave2Vec2 bundle available through torchaudio.  Note that this method includes,
#by default, the feature dimension for <star> token.  

from torchaudio.pipelines import MMS_FA as bundle

model = bundle.get_model()
model.to(device)

tokenizer = bundle.get_tokenizer()
aligner = bundle.get_aligner()

In [ ]:
#Check the tokenizer's mapping of the normalized characters to integers.
print(bundle.get_dict())

In [ ]:
#Define a utility function that performces the forced alignment with the model, tokenizer, and
#aligner instantiated above.

def compute_alignments(waveform: torch.Tensor, transcript: List[str]):
    with torch.inference_mode():
        emission, _ = model(waveform.to(device))
        token_spans = aligner(emission[0], tokenizer(transcript))
    return emission, token_spans

In [ ]:
#Normalize the transcript by first using uroman (in BASH, pre-process step) to romanize and then regex (imported above as re) 
#to remove non-alphabets and punctuations.

def normalize_uroman(text):
    text = text.lower()  #makes all input text lowercase
    text = text.replace("’", "'") #replaces apostrophe characters with parsable ones 

    # Normalize accented characters into ASCII (NFD decomposes characters into base + accent)
    text = unicodedata.normalize('NFD', text)
    text = ''.join([char for char in text if unicodedata.category(char) != 'Mn'])

    #Apply regex to remove non a-z characters
    text = re.sub("([^a-z' ])", " ", text) #replaces any character that isn't lowercase alphabet, space, or apostraphe with a blank space
    text = re.sub(' +', ' ', text) #replaces one or more consecutive spaces in the string with a single space
    return text.strip() #removes any leading or trailing whitespace characters from string

#Note: this normalization process will remove numbers.  If there are any numbers, they should be
#converted to words instead of numerals prior to normalization.

In [ ]:
#Read in the romanized transcript and normalize it using the function defined above.
def normalization(transcriptdir):
    with open(transcriptdir, "r") as f:
        text_normalized = " ".join(normalize_uroman(line).strip() for line in f if line.strip())
        #print(text_normalized)
    return text_normalized

In [ ]:
#create a function that will add stars between words in the transcript 
def sneetches(normd_text):
    words = normd_text.split() #split text into words
    return " * ".join(words)  # Join words with "*"

In [ ]:
#Now, in preparation for the alignment, let's initialize a function that computes the 
#average score weighted by the span length.  This score can be understood as the model's confidence
#in the transcript/value it assigns to each span.

def _score(spans):
    return sum(s.score * len(s) for s in spans) / sum(len(s) for s in spans)

In [ ]:
#Define another function that will hold all the relevant information we need to output from our
#forced alignment of the .wav file

def word_info(waveform, spans, num_frames, transcript, wav_file, sample_rate=bundle.sample_rate):
    ratio = waveform.size(1) / num_frames
    t_start = int(ratio * spans[0].start)
    t_stop = int(ratio * spans[-1].end)
    return(f"{wav_file},{transcript},{_score(spans):.2f}, {t_start / sample_rate:.3f},{t_stop / sample_rate:.3f}")

In [ ]:
#Define another function that will compute the threshold for each speaker 
#The threshold is the segment duration (s) beyond which a wildcard segment is considered to contain an error and removed from consideration

def thresholding(token_spans, waveform, num_frames, sample_rate):

    ratio = waveform.size(1) / num_frames # Scaling factor to convert frames to time

    wildcards = token_spans[1::2] #Extract only wildcard token spans
    wc_durations = [(wc[-1].end - wc[0].start) * ratio / sample_rate for wc in wildcards]  # Compute durations using list comprehension

    if not wc_durations:
        raise ValueError("No wildcard durations found. Check token_spans input.")

    log_wc_durations = np.log(wc_durations) #data not normally distributed; apply log transform
    
    mean_log_wc_duration = np.mean(log_wc_durations)
    stdev_log_wc_duration = np.std(log_wc_durations, ddof=1)

    threshold = mean_log_wc_duration + (1* stdev_log_wc_duration)
    print(f"Threshold is {threshold:.4f} given mean of {mean_log_wc_duration:.4f} and stdev of {stdev_log_wc_duration:.4f}")

    return threshold

In [ ]:
#Define a function that identifies the longest continguous regions in the aligned transcripts while ensuring that sections corresponding to wildcards (*)
#don't have anomalously large durations.

def build_regions(token_spans, transcript, waveform, threshold, num_frames, sample_rate, index_range):

    start_idx, end_idx = index_range #extract bounds of ISI
    ratio = waveform.size(1) / num_frames  # Convert frames to time
    
    regions = [] #holds valid regions
    current_region = [] #stores tokens constituting the current region
    skip_next = False

    #for i, (t_spans, token) in enumerate(zip(token_spans, transcript)):
    for i in range(start_idx, min(end_idx, len(token_spans))): #process only within ISI index range

        t_spans, token = token_spans[i], transcript[i]
        
        start_time, end_time = t_spans[0].start * ratio / sample_rate, t_spans[-1].end * ratio / sample_rate #get start/end times in seconds
        duration = end_time - start_time #compute total duration of token in seconds
        log_duration = np.log(duration)
        #print(duration)

        if skip_next:
            skip_next = False
            current_region = [] #redundant but just makes sure current region is reset
            continue #don't add the current region and go to the next index

        
        if i % 2 != 0 and token == "*" and log_duration > threshold: 
            #append the current region and start a new one if wildcard region duration exceeds the threshold
            
            if current_region: #quick check to ensure the current region list isn't empty 
                regions.append(current_region)
                skip_next = True
                current_region = [] #reset current region/start to build new region
                continue #skip adding wildcard to the new region
                
        current_region.append((token, start_time, end_time, i)) #if this region isn't being skipped due to error in current or preceding region, append token

    if current_region:
        regions.append(current_region)

    return regions       

In [ ]:
#Define a function that will build regions for pre-defined 'chunks' of the force-aligned audio (i.e.,
#the inter-switch intervals (our ISI's).

def build_regions_by_isi(token_spans, transcript, waveform, threshold, num_frames, sample_rate, isi_indices):
    regions_by_isi = []

    for start_idx, end_idx in isi_indices: #use pre-defined index pairs
        regions = build_regions(token_spans, transcript, waveform, threshold, num_frames, sample_rate, (start_idx, end_idx))
        regions_by_isi.append(regions)

    return regions_by_isi


In [ ]:
#Define a function that compares two lists of regions, finds and returns the longest contiguous overlapping subsequence between two lists

def find_longest_overlap(mixed_regions, single_regions):
    max_length = 0  # Tracks the max length of common substring
    best_match = []
    best_mixed_bounds = None
    best_single_bounds = None
    best_og_indices = None
    
    #iterate through each pair of regions (doesn't assume the input lists contain equal numbers of sublists)
    for mixed_idx, mixed_region in enumerate(mixed_regions): 
        for single_idx, single_region in enumerate(single_regions): 

            #Extract words and token bounds from the tuples
            mixed_words = [word_tuple[0] for word_tuple in mixed_region]
            single_words = [word_tuple[0] for word_tuple in single_region]
            mixed_token_bounds = [(token_tuple[1], token_tuple[2]) for token_tuple in mixed_region]
            single_token_bounds = [(token_tuple[1], token_tuple[2]) for token_tuple in single_region]
            mixed_og_indices = [index_tuple[3] for index_tuple in mixed_region]
            #single_og_indices = [index_tuple[3] for index_tuple in single_region]
            #print("Mixed Region:", mixed_words)
            #print("Single Region:", single_words)

            #create a table for dynamic programming; initialized with zeros and len + 1 to simplify boundary conditions
            dp = [[0] * (len(single_words) + 1) for _ in range(len(mixed_words) + 1)] #underscore b/c not interested in results of loop, just need certain # of iterations

            local_max_length = 0
            local_mixed_end = 0  # local ending index in mixed_words
            local_single_end = 0 # Local ending index in single_words

            for i in range(1, len(mixed_words) + 1):
                for j in range(1, len(single_words) + 1):
                        if mixed_words[i - 1] == single_words[j - 1]:  #compare tokens at each position; i - 1 b/c lists are zero indexing in Python
                            dp[i][j] = dp[i - 1][j - 1] + 1 #if equivalent, extend previous match
                
                            if dp[i][j] > local_max_length: #update max length and end index of local match
                                local_max_length = dp[i][j]
                                local_mixed_end = i 
                                local_single_end = j

            #Check dp table for this pair of regions
            #print("DP Table:", dp)
            
            #Slice mixed_words to get longest match for this pair 
            if local_max_length > 0:
                longest_match = mixed_words[local_mixed_end - local_max_length:local_mixed_end]
            else: 
                longest_match = []
            print(f"Longest match for Mixed Region {mixed_idx} & Single Region {single_idx}: {longest_match}")

            #If the current pair's match is the best overall, update global bests
            if local_max_length > max_length:
                max_length = local_max_length
                best_match = longest_match

                #These are local indices relative to the current region
                best_mixed_indices = (local_mixed_end - local_max_length, local_mixed_end)
                best_single_indices = (local_single_end - local_max_length, local_single_end)

                #Convert local indices to absolute time bounds using token bounds lists
                best_mixed_bounds = (mixed_token_bounds[best_mixed_indices[0]][0],
                                     mixed_token_bounds[best_mixed_indices[1]-1][1])
                best_single_bounds = (single_token_bounds[best_single_indices[0]][0],
                                      single_token_bounds[best_single_indices[1]-1][1])

                #Extract original (not local) indices
                best_og_indices = (mixed_og_indices[best_mixed_indices[0]], mixed_og_indices[best_mixed_indices[1]-1])
                #best_single_og_indices = (single_og_indices[best_single_indices[0], single_og_indices[best_single_indices[1]-1])
                        

            #ipdb.set_trace()

    return best_match, best_mixed_bounds, best_single_bounds, best_og_indices

In [ ]:
#Define a function to process an individual audio file and return transcript regions

def process_audio_file(wav_file, text_normalized, indices):
    #run a series of checks to ensure the wavfile is able to be read in properly
    current_file = str(wav_file) #torchaudio.load expects a path-like object but formatted as a string
    
    print(f"Processing file: {current_file}")
    wav_path = Path(wav_file)  # Convert string path to a Path object
    print(f"Current file exists? {wav_path.exists()}")

    import mimetypes
    print(mimetypes.guess_type(wav_file))
    print(os.access(wav_file, os.R_OK))

   #load in the wavfile using torchaudio and instantiate two new variables to hold the info
    waveform, sample_rate = torchaudio.load(current_file)
    print(sample_rate)

    #resample the audiofile at a desired sample rate and set sample_rate (whatever the audio was originally sampled at) to the desired sample rate for processing
    waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)
    sample_rate = bundle.sample_rate
    assert sample_rate == bundle.sample_rate #checks that the value of sample_rate has not been inadvertently changed or that the bundle.sample_rate attribute has not been modified elsewhere in the code.
    print(type(sample_rate)) #checks that sample_rate is a numeric type (int or float)
    """
        Details: Many pre-trained models in torchaudio expect audio input at a specific sampling rate (e.g., 16 kHz or 44.1 kHz).  
        If the audio file’s original sample rate (sample_rate) differs from the model’s required rate (e.g., bundle.sample_rate), 
        resampling ensures that the input is compatible.
    """

    #calls functions to normalize, add wildcards (if desired), and tokenize the written transcripts
    if starsupon:
        starredtxt = sneetches(text_normalized) #call earlier function to add stars between words
        transcript = starredtxt.split() #divide the transcript into a list of individual words
        print("Starred transcript:", transcript)
        tokens = tokenizer(transcript)
    else: 
        transcript = text_normalized.split() #divide the transcript into a list of individual words
        tokens = tokenizer(transcript) #tokenize the transcript

    #print("Tokenized transcript:", tokens) #checks that transcript var has been appropriately tokenized

    #Compute alignments
    #Check if stereo and, if so, convert to mono since wav2vec2 requries mono audio input
    if waveform.shape[0] > 1:  
        waveform = waveform.mean(dim=0, keepdim=True)  
    
    print("Waveform shape:", waveform.shape) #checks the batch dimensions, should be two: channels, time

    emission, token_spans = compute_alignments(waveform, transcript)
    num_frames = emission.size(1)

    #Identify and build transcript regions
    threshold = thresholding(token_spans, waveform, num_frames, sample_rate)
    regions = build_regions_by_isi(token_spans, transcript, waveform, threshold, num_frames, sample_rate, indices)

    
    return waveform, sample_rate, regions, transcript, token_spans, os.path.basename(current_file).split('.')[0], num_frames
    

In [ ]:
#Define a function that takes in the waveform (entire audio as a tensor), sample rate, an ISI region,
#and start/end times from the overlap function and returns slice of waveform corresponding to overlap region.  

def extract_audio_segment(waveform, sample_rate, bounds):

    #defensive coding: ensure indices are valid
    if bounds is None:
        return None

    #Use the bounds to extract the audio segment
    start_time, end_time =  bounds #bounds is a tuple (start_time, end_time)

    #convert times to sample indices
    start_sample = int(start_time * sample_rate)
    end_sample = int(end_time * sample_rate)

    #Slice the waveform (assuming wf shape is [channels, samples])

    audio_segment = waveform[:, start_sample:end_sample] #":" means all rows and range of columns from start(inclusive) to stop (exclusive)
    return audio_segment

In [ ]:
#Function to save alignments to csv 

def save_alignments_to_csv(name, transcript, token_spans, waveform, wav_file, num_frames, output_dir):
    
    #Initialize csv file to be used to hold the data
    #Define the filename for the new CSV file
    output_filename = os.path.join(output_dir, f"{name}.csv")
    print(output_filename)

    #Define the header row
    header = ["file", "transcription", "score", "start", "stop"]
    
    #Prepare data for output
    outputlist = [] 

    for i in range(0, len(transcript)):
        word_entry = word_info(waveform, token_spans[i], num_frames, transcript[i], wav_file)
        #print(f"Index {i}: {word_entry}")
        word_entry_list = word_entry.split(",") #ensure entries are split by commas and not spaces
        #outputlist.append(word_entry_list + "\n")
        outputlist.append(word_entry_list)

    print("Example entry:", outputlist[0])

    #ipdb.set_trace()
    
    #Create and write the header to the CSV file
    with open(output_filename, mode="w", newline="", encoding="utf-8") as file: #creates and opens the output file in write mode and writes the header row
        writer = csv.writer(file, quoting=csv.QUOTE_ALL) # Forces all fields to be enclosed in quotes to ensure file paths with spaces remain intact 
        writer.writerow(header)
        writer.writerows([entry for entry in outputlist])
        #writer.writerows([entry.split() for entry in outputlist])

    print(f"CSV file '{output_filename}' created successfully.")

In [ ]:
#Function to write longest regions to csv

def save_longest_regions_to_csv(name, longest_regions, longest_bounds, longest_indices, outputdir):

    #Initialize csv file to be used to hold the data
    #Define the filename for the new CSV file
    output_filename = os.path.join(outputdir, f"{name}_longestregions.csv")
    print(output_filename)

    #Define the header row
    header = ["file", "region_num", "region_length", "start_idx", "stop_idx", "mixed_Tstart", "mixed_Tstop", "single_Tstart", "single_Tstop", "region_content"]

    with open(output_filename, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(header)
    
        for i, (region, bounds, indices) in enumerate(zip(longest_regions, longest_bounds, longest_indices)):
            region_length = len(region)
            mixed_bounds, single_bounds = bounds
            mixed_start, mixed_stop = mixed_bounds
            single_start, single_stop = single_bounds
            start_idx, stop_idx = indices
            words = ' '.join(region)
    
            # Construct row
            row = [name, i, region_length, start_idx, stop_idx, mixed_start, mixed_stop, single_start, single_stop, words]
            writer.writerow(row) 
        

In [ ]:
#Determine the paragraph language from the file stem

def get_language(filename):
    if "LangAOnly" in filename or "LangADefault" in filename:
        return "LangA"
    elif "LangBOnly" in filename or "LangBDefault" in filename:
        return "LangB"
    else:
        return None

In [ ]:
def strip_suffix(stem, suffixes):
    for suffix in suffixes:
        if stem.endswith(suffix):
            return stem[: -len(suffix)]
    return stem  # Return unchanged if no suffix matched

In [ ]:
def parse_filename(name):
    """
    Parses audio filenames to extract participant, language, condition, and paragraph.
    Note: For backwards compatibility with prior work, this function supports 
    an additional legacy naming convention (YA scheme) alongside the 
    standard format documented in the README (OA scheme).
    """
    # Try YA scheme: e.g., "b2_sdp1" → b2, s (Span), d (default), p1
    match = re.match(r"(?P<participant>[a-z0-9]+)_(?P<lang>[se])(?P<cond>[do])p(?P<num>\d+)", name, re.IGNORECASE)
    if match:
        lang_map = {'s': 'LangA', 'e': 'LangB'}
        cond_map = {'d': 'default', 'o': 'single'}
        lang = lang_map.get(match['lang'].lower())
        cond = cond_map.get(match['cond'].lower())
        pgph = f"Pgph{match['num']}"
        return {"participant": match['participant'], "lang": lang, "cond": cond, "pgph": pgph}

    # Try OA scheme: e.g., "B102R_DL_Switch_Pgph1_SpanOnly"
    match = re.search(r"(?P<pgph>Pgph\d+).*_(?P<lang>LangA|LangB)(Default|Only)", name)
    if match:
        lang = match['lang']
        cond = 'mixed' if 'Default' in name else 'single'
        pgph = match['pgph']
        participant = name.split("_")[0]
        return {"participant": participant, "lang": lang, "cond": cond, "pgph": pgph}

    return None

In [ ]:
#Processes all paired audio files (from single and mixed conditions) in 
#specified set of directories; transcripts should be normalized before 
#passing into this function

def process_paired_audio_files(single_dir, mixed_dir, paragraph_keyword, 
                                mixed_transcript, single_transcript,
                                output_csv_dir, audio_output_path, output_regions_dir):

    #Build dictionaries keyed by (base_stem, lang)
    single_files = {}
    for f in single_dir.glob("*.wav"):
        info = parse_filename(f.stem)
        if info and info["cond"] == "single" and info["pgph"] == paragraph_keyword:
            key = (info["participant"], info["lang"])
            single_files[key] = f

    mixed_files = {}
    for f in mixed_dir.glob("*.wav"):
        info = parse_filename(f.stem)
        if info and info["cond"] == "mixed" and info["pgph"] == paragraph_keyword:
            key = (info["participant"], info["lang"])
            mixed_files[key] = f

    shared_keys = set(single_files) & set(mixed_files)
    print("Single file keys:", single_files.keys())
    print("Mixed file keys:", mixed_files.keys())
    print("Shared keys:", shared_keys)

    for (participant, lang) in shared_keys:
        wav_single = single_files[(participant, lang)]
        wav_mixed = mixed_files[(participant, lang)]

        print(f"Processing participant {participant} ({lang})")

        #Choose correct indices
        if lang == "LangA" and paragraph_keyword == "Pgph1":
            indices = LangA_P1_indices
        elif lang == "LangA" and paragraph_keyword == "Pgph2":
            indices = LangA_P2_indices
        elif lang == "LangB" and paragraph_keyword == "Pgph1":
            indices = LangB_P1_indices
        else:
            indices = LangB_P2_indices

        #Alignment and segmentation
        mixed_waveform, mixed_samplerate, mixed_regions, local_mixed_transcript, mixed_spans, mixed_name, mixed_frames = process_audio_file(wav_mixed, mixed_transcript, indices)
        single_waveform, single_samplerate, single_regions, local_single_transcript, single_spans, single_name, single_frames = process_audio_file(wav_single, single_transcript, indices)

        #Extract longest contiguous valid region
        longest_regions, longest_bounds, longest_indices = extract_longest_overlap_per_isi(mixed_regions, single_regions)

        for i, bounds in enumerate(longest_bounds):
            mixed_bounds, single_bounds = bounds
            if mixed_bounds and single_bounds:
                mixed_segment = extract_audio_segment(mixed_waveform, mixed_samplerate, mixed_bounds)
                mixed_output_filename = os.path.join(audio_output_path, f"{mixed_name}_longestoverlap_ISI_{i}_test.wav")
                torchaudio.save(mixed_output_filename, mixed_segment, mixed_samplerate)

                single_segment = extract_audio_segment(single_waveform, single_samplerate, single_bounds)
                single_output_filename = os.path.join(audio_output_path, f"{single_name}_longestoverlap_ISI_{i}_test.wav")
                torchaudio.save(single_output_filename, single_segment, single_samplerate)

        #Save alignment and region metadata
        save_alignments_to_csv(mixed_name, local_mixed_transcript, mixed_spans, mixed_waveform, wav_mixed, mixed_frames, output_csv_dir)
        save_alignments_to_csv(single_name, local_single_transcript, single_spans, single_waveform, wav_single, single_frames, output_csv_dir)
        save_longest_regions_to_csv(mixed_name, longest_regions, longest_bounds, longest_indices, output_regions_dir)

In [ ]:
#Define a function that will loop over the ISIs to enable comparison of the regions.

def extract_longest_overlap_per_isi(mixed_regions, single_regions):

    longest_regions = [] #will store lists of match_words
    longest_bounds = [] #will store lists of start/end times for each region per condition (single vs. mixed)
    longest_indices = [] #will store lists of indices for each each region per condition
    
    for i in range(len(mixed_regions)): 
        longest_match, mixed_bounds, single_bounds, best_indices = find_longest_overlap(mixed_regions[i], single_regions[i])
        print("Longest Contiguous Overlapping Region:", longest_match)
        longest_regions.append(longest_match)
        longest_bounds.append([mixed_bounds, single_bounds])
        longest_indices.append(best_indices)

    return longest_regions, longest_bounds, longest_indices

In [ ]:
#Define input audio and transcript paths if processing files individually  

#Specify the two audio files

wav_file_mixed = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Audio Files\AudioFiles_EngMatrix\B112_DL_Switch_Pgph2_EngMatrix.wav"
wav_file_single = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Audio Files\AudioFiles_EngOnly\B112_DL_Switch_Pgph2_EngOnly.wav"

#And the two transcript files
mixed_transcriptdir = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Transcript Text Files\EngMatrix_P2_Dancer_Base.txt"
single_transcriptdir = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Transcript Text Files\EngOnly_P2_Dancer_Base.txt"

In [ ]:
#Process two paired audio files (forced align, build regions, compare regions)

#Boolean that sets whether to intercalate wildcards in target text
starsupon = True 

#Define ISI indices
LangA_P1_indices = [(0,23), (29,63), (69, 105), (111, 145), (151,161), (167, 233), (239, 273), (279, 295), (301, 337), (343, 361), (367, 388)]
LangB_P1_indices = [(0,23), (29,65), (71, 107), (113, 145), (151,161), (167, 231), (237, 271), (277, 295), (301, 337), (343, 363), (369, 390)]
LangA_P2_indices = [(0,47), (53, 81), (87, 97), (103, 117), (123, 139), (145, 193), (199, 227), (233, 293), (299, 307), (313, 359), (365, 398)]
LangB_P2_indices = [(0,49), (55, 83), (89, 99), (105, 117), (123, 137), (143, 191), (197, 227), (233, 289), (295, 303), (309, 357), (363, 390)]

#Normalize the transcript text
mixed_normalized = normalization(mixed_transcriptdir)
single_normalized = normalization(single_transcriptdir)
mixed_waveform, mixed_samplerate, mixed_regions, mixed_transcript, mixed_spans, mixed_name, mixed_frames = process_audio_file(wav_file_mixed, mixed_normalized, LangA_P1_indices)
single_waveform, single_samplerate, single_regions, single_transcript, single_spans, single_name, single_frames = process_audio_file(wav_file_single, single_normalized, LangA_P1_indices)

In [ ]:
longest_regions, longest_bounds, longest_indices = extract_longest_overlap_per_isi(mixed_regions, single_regions)

#Set output path for extracted audio 
audio_output_path = "C:\\Users\\jjsar\\OneDrive\\Documents\\PostDoc\\Transcription and Forced Alignment\\Audio Files\\Extracted Audio\\RegionsTesting"

#for ISI in longest_bounds:
for i, bounds in enumerate(longest_bounds):
    mixed_bounds, single_bounds = bounds #unpack the bounds tuple

    #only process if both bounds are valid
    if mixed_bounds is not None and single_bounds is not None:

        #extract the mixed audio segment
        mixed_segment = extract_audio_segment(mixed_waveform, mixed_samplerate, mixed_bounds)
        #save mixed segment to file
        mixed_output_filename = os.path.join(audio_output_path, f"{mixed_name}_longestoverlap_ISI_{i}_testing.wav")
        torchaudio.save(mixed_output_filename, mixed_segment, mixed_samplerate)
        #torchaudio.save(f"audio_output_path\\{mixed_name}_longestoverlap_ISI_{i}.wav", mixed_segment, mixed_samplerate)

        #Extract the single audio segment
        single_segment = extract_audio_segment(single_waveform, single_samplerate, single_bounds)
        #Save the single segment to file
        single_output_filename = os.path.join(audio_output_path, f"{single_name}_longestoverlap_ISI_{i}_testing.wav")
        torchaudio.save(single_output_filename, single_segment, single_samplerate)
        #torchaudio.save(f"ISI_{i}_single_longestoverlap.wav", single_segment, single_samplerate)

        #Print the shape for debugging
        print("Extracted audio segment shape:", mixed_segment.shape)
        print("Extracted audio segment shape:", single_segment.shape)

In [ ]:
for region in longest_regions:
        if region:  # Check if region is not empty
            #print(region[0][0])  # Print the first token of the first tuple in the region
            print("Region content:", region)

In [ ]:
#Save alignments: Mixed Language Audio Files

output_filepath = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\FA_Output_EngMatrix\Test_Data_Regions\Testing"

save_alignments_to_csv(mixed_name, mixed_transcript, mixed_spans, mixed_waveform, wav_file_mixed, mixed_frames, output_filepath)

In [ ]:
#Save alignments: Single Language Audio Files

output_filepath = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\FA_Output_EngMatrix\Test_Data_Alignments\TestData"

save_alignments_to_csv(single_name, single_transcript, single_spans, single_waveform, wav_file_single, single_frames, output_filepath)

In [ ]:
output_filepath = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\FA_Output_EngMatrix\Test_Data_Alignments\TestData"

save_longest_regions_to_csv(mixed_name, longest_regions, longest_bounds, longest_indices, output_filepath)

In [ ]:
#Define input audio and transcript paths if batch processing files

#Specify the two folders of audio files

single_dir = r"C:\Users\jjsar\OneDrive - Northwestern University\PostDoc\Transcription and Forced Alignment\YoungAdult_Data\AudioFiles\Pgphs3and4\SpanOnly"
mixed_dir = r"C:\Users\jjsar\OneDrive - Northwestern University\PostDoc\Transcription and Forced Alignment\YoungAdult_Data\AudioFiles\Pgphs3and4\SpanMatrix"

#Specify the transcript files
Pgph1_mixed_transcriptdir = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Transcript Text Files\EngMatrix_P1_Marathon_Base.txt"
Pgph1_single_transcriptdir = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Transcript Text Files\EngOnly_P1_Marathon_Base.txt"
Pgph2_mixed_transcriptdir = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Transcript Text Files\EngMatrix_P2_Dancer_Base.txt"
Pgph2_single_transcriptdir = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\Transcript Text Files\EngOnly_P2_Dancer_Base.txt"

#Specify the output directories
audio_output_path = r"C:\Users\jjsar\OneDrive\Documents\PostDoc\Transcription and Forced Alignment\YoungAdult_Data\AudioFiles\YA_Extracted_Audio\YA_EngExtracts\Pgphs3and4"
output_csv_dir = r"C:\Users\jjsar\OneDrive - Northwestern University\PostDoc\Transcription and Forced Alignment\YoungAdult_Data\YA_alignments\Pgphs3and4"
output_regions_dir = r"C:\Users\jjsar\OneDrive - Northwestern University\PostDoc\Transcription and Forced Alignment\YoungAdult_Data\YA_regions\Pgphs3and4"

In [ ]:
#let's process the data
## USER NOTES: NEED TO SET TRANSCRIPT TEXT TO CORRECT PGPHS THAT CORRESPONDS TO PGPH KEYWORD

#Boolean that sets whether to intercalate wildcards in target text
starsupon = True 

#Define ISI indices
LangA_P1_indices = [(0,23), (29,63), (69, 105), (111, 145), (151,161), (167, 233), (239, 273), (279, 295), (301, 337), (343, 361), (367, 388)]
LangB_P1_indices = [(0,23), (29,65), (71, 107), (113, 145), (151,161), (167, 231), (237, 271), (277, 295), (301, 337), (343, 363), (369, 390)]
LangA_P2_indices = [(0,47), (53, 81), (87, 97), (103, 117), (123, 139), (145, 193), (199, 227), (233, 293), (299, 307), (313, 359), (365, 398)]
LangB_P2_indices = [(0,49), (55, 83), (89, 99), (105, 117), (123, 137), (143, 191), (197, 227), (233, 289), (295, 303), (309, 357), (363, 390)]

#Normalize the transcript text
mixed_normalized = normalization(Pgph1_mixed_transcriptdir)
single_normalized = normalization(Pgph1_single_transcriptdir)

#Batch process audio files

paragraph_keyword = "Pgph1"
process_paired_audio_files(Path(single_dir), Path(mixed_dir), paragraph_keyword, 
                                mixed_normalized, single_normalized,
                                Path(output_csv_dir), Path(audio_output_path), Path(output_regions_dir))
